# 📅 Notebook 05 — Daily Revenue Forecast (30-Day Series)

**Model**: `daily_revenue_forecast.pkl`  
**Algorithm**: ARIMA (AutoRegressive Integrated Moving Average)  
**Target**: Array of 30 daily revenue forecasts for next month  
**Input**: Historical daily revenue time series (engineered from order data)

In [ ]:
# What: Import ARIMA from statsmodels and supporting libraries
# Why: ARIMA is the standard method for univariate time-series forecasting
# Method: statsmodels ARIMA + pandas for series handling + joblib for saving
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt
import joblib, os, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

DATA_PATH    = '../data'
MODELS_PATH  = '../models'
os.makedirs(MODELS_PATH, exist_ok=True)

# Department avg price proxy (Rs.) — consistent across all notebooks
DEPT_PRICE = {1:350,2:200,3:250,4:150,5:800,6:400,7:180,8:600,9:300,
              10:200,11:500,12:700,13:350,14:280,15:200,16:300,17:450,
              18:500,19:220,20:400,21:200}
print('Config loaded ✅')

In [ ]:
# What: Load orders and order-product data
# Why: Need order timing (order_dow, order_number) and item counts to build daily revenue
# Method: pandas read_csv for orders and op_train files
orders   = pd.read_csv(f'{DATA_PATH}/orders.csv')
op_train = pd.read_csv(f'{DATA_PATH}/order_products__train.csv')
products    = pd.read_csv(f'{DATA_PATH}/products.csv')
departments = pd.read_csv(f'{DATA_PATH}/departments.csv')

print(f'orders: {orders.shape} | op_train: {op_train.shape}')

In [ ]:
# What: Build item-level table with department and revenue
# Why: Must know each item's department to assign correct price
# Method: Merge chain op_train → products → departments → orders
products_dept = products.merge(departments, on='department_id', how='left')
order_items = op_train.merge(products_dept[['product_id','department_id']], on='product_id', how='left')
order_items = order_items.merge(
    orders[['order_id','order_number','order_dow']],
    on='order_id', how='left'
)
order_items['price']    = order_items['department_id'].map(DEPT_PRICE).fillna(200)
order_items['revenue']  = order_items['price']
order_items['month_num'] = np.ceil(order_items['order_number'] / 3).astype(int)

print('Item table ready:', order_items.shape)

In [ ]:
# What: Build a daily revenue time series using (month_num × 7) + order_dow as synthetic day index
# Why: ARIMA needs a sequential daily time series — we simulate it from order data
# Method: Assign each order a synthetic day = (month_num - 1) * 7 + order_dow + 1
order_items['synthetic_day'] = (order_items['month_num'] - 1) * 7 + order_items['order_dow'] + 1

daily_rev = order_items.groupby('synthetic_day')['revenue'].sum().reset_index()
daily_rev.columns = ['day', 'revenue']
daily_rev.sort_values('day', inplace=True)
daily_rev.reset_index(drop=True, inplace=True)

print(f'Daily time series length: {len(daily_rev)} days')
print(f'Revenue range: Rs. {daily_rev["revenue"].min():,.0f} → Rs. {daily_rev["revenue"].max():,.0f}')
print(daily_rev.head(10))

In [ ]:
# What: Plot the daily revenue time series
# Why: Visually inspect trend, seasonality, and volatility before fitting ARIMA
# Method: Line chart of synthetic daily revenue
plt.figure(figsize=(14, 4))
plt.plot(daily_rev['day'], daily_rev['revenue'], color='darkorange', linewidth=0.9, alpha=0.9)
plt.fill_between(daily_rev['day'], daily_rev['revenue'], alpha=0.1, color='darkorange')
plt.title('Synthetic Daily Revenue Time Series', fontsize=14, fontweight='bold')
plt.xlabel('Synthetic Day Index')
plt.ylabel('Daily Revenue (Rs.)')
plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/chart_daily_ts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Time series plot saved ✅')

In [ ]:
# What: Augmented Dickey-Fuller test for stationarity
# Why: ARIMA requires the series to be stationary (no unit root); determines if differencing needed
# Method: adfuller on revenue series — p < 0.05 means stationary
rev_series = daily_rev['revenue'].values
adf_result = adfuller(rev_series)
print('ADF Test Results:')
print(f'  ADF Statistic : {adf_result[0]:.4f}')
print(f'  p-value       : {adf_result[1]:.6f}')
print(f'  Series is {"stationary ✅" if adf_result[1] < 0.05 else "non-stationary — will difference"}')

In [ ]:
# What: Plot ACF and PACF to determine ARIMA p, d, q parameters
# Why: ACF shows MA lags, PACF shows AR lags — key for choosing ARIMA order
# Method: statsmodels plot_acf and plot_pacf on stationary (differenced if needed) series
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Use first-difference if non-stationary
series_to_plot = pd.Series(rev_series).diff().dropna() if adf_result[1] >= 0.05 else pd.Series(rev_series)

plot_acf(series_to_plot,  ax=axes[0], lags=40, color='steelblue')
axes[0].set_title('ACF — Autocorrelation Function')

plot_pacf(series_to_plot, ax=axes[1], lags=40, color='darkorange', method='ywm')
axes[1].set_title('PACF — Partial Autocorrelation Function')

plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/chart_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()
print('ACF/PACF plots saved ✅')

In [ ]:
# What: Split time series into train and test for validation
# Why: Evaluate how well ARIMA forecasts on unseen recent days
# Method: Use last 30 days as test, all prior as training
TEST_DAYS  = 30
train_rev  = rev_series[:-TEST_DAYS]
test_rev   = rev_series[-TEST_DAYS:]
print(f'Train days: {len(train_rev)} | Test days: {len(test_rev)}')

In [ ]:
# What: Fit ARIMA(5,1,2) model on training series
# Why: p=5 captures weekly pattern (7-day cycle uses nearby lags); d=1 handles non-stationarity; q=2 for MA smoothing
# Method: statsmodels ARIMA.fit() — automatically handles differencing internally
PORDER = (5, 1, 2)  # (AR order, differencing, MA order)

arima_model = ARIMA(train_rev, order=PORDER)
arima_result = arima_model.fit()

print(f'ARIMA{PORDER} fitted ✅')
print(arima_result.summary().tables[0])

In [ ]:
# What: Generate 30-day test forecast and compute RMSE
# Why: Validate ARIMA accuracy on held-out test period before saving
# Method: model.forecast(steps=30) produces point forecasts with confidence intervals
forecast_test = arima_result.forecast(steps=TEST_DAYS)

from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(test_rev, forecast_test))
mape = np.mean(np.abs((test_rev - forecast_test) / np.clip(test_rev, 1, None))) * 100

print(f'Forecast RMSE : Rs. {rmse:,.2f}')
print(f'MAPE          : {mape:.2f}%')

In [ ]:
# What: Plot actual vs forecasted daily revenue on test period
# Why: Visual confirmation that ARIMA captures daily revenue patterns
# Method: Overlay actual test series and forecast on same axes
days_total = np.arange(len(rev_series))
days_test  = days_total[-TEST_DAYS:]

plt.figure(figsize=(14, 5))
plt.plot(days_total[:-TEST_DAYS], train_rev, color='steelblue',  label='Training', linewidth=1)
plt.plot(days_test,              test_rev,   color='green',      label='Actual (Test)', linewidth=1.5)
plt.plot(days_test,              forecast_test, color='darkorange', label='ARIMA Forecast',
         linewidth=2, linestyle='--')
plt.axvline(x=days_test[0], color='gray', linestyle=':', linewidth=1)
plt.title(f'ARIMA Daily Revenue Forecast vs Actual\nRMSE: Rs. {rmse:,.0f} | MAPE: {mape:.1f}%',
          fontsize=13, fontweight='bold')
plt.xlabel('Synthetic Day Index')
plt.ylabel('Daily Revenue (Rs.)')
plt.legend()
plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/chart_arima_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('Forecast chart saved ✅')

In [ ]:
# What: Refit ARIMA on FULL series for production forecasting
# Why: Test split was for validation only; the final model uses all available data
# Method: Re-train on full rev_series before saving
final_arima = ARIMA(rev_series, order=PORDER)
final_result = final_arima.fit()

# Generate 30-day future forecast
future_forecast = final_result.forecast(steps=30)
print('30-Day Revenue Forecast (Rs.):')
for day_i, rev_i in enumerate(future_forecast, 1):
    print(f'  Day {day_i:2d}: Rs. {rev_i:>10,.2f}')

In [ ]:
# What: Save the fitted ARIMA model and forecast metadata as .pkl
# Why: ARIMA fit result must be persisted with forecast array for API use
# Method: Save bundle dict: fitted model result + 30-day forecast array + metadata
forecast_bundle = {
    'model_result':     final_result,             # fitted ARIMA for future reforecasting
    'forecast_30_days': future_forecast.tolist(), # pre-computed 30-day forecast (Rs.)
    'arima_order':      PORDER,                   # (p,d,q) for documentation
    'series_length':    len(rev_series),          # history length
    'avg_daily':        float(np.mean(future_forecast)),
    'total_forecast':   float(np.sum(future_forecast))
}
joblib.dump(forecast_bundle, f'{MODELS_PATH}/daily_revenue_forecast.pkl')
print('✅ Saved: models/daily_revenue_forecast.pkl')
print(f'   Size: {os.path.getsize(f"{MODELS_PATH}/daily_revenue_forecast.pkl") / 1024:.1f} KB')
print(f"\n📊 Forecasted Monthly Total : Rs. {forecast_bundle['total_forecast']:,.2f}")
print(f"📊 Avg Daily Revenue        : Rs. {forecast_bundle['avg_daily']:,.2f}")

In [ ]:
# What: Verify .pkl loads correctly and forecast is usable
# Why: Confirm API can load model and retrieve the 30-day forecast array
# Method: Load bundle, extract forecast, plot the 30-day output chart
loaded = joblib.load(f'{MODELS_PATH}/daily_revenue_forecast.pkl')
forecast_arr = np.array(loaded['forecast_30_days'])

plt.figure(figsize=(13, 4))
day_labels = [f'Day {i}' for i in range(1, 31)]
plt.fill_between(range(1, 31), forecast_arr, alpha=0.3, color='darkorange')
plt.plot(range(1, 31), forecast_arr, marker='o', color='darkorange', linewidth=2, markersize=4)
plt.title('Next Month — 30-Day Daily Revenue Forecast', fontsize=14, fontweight='bold')
plt.xlabel('Day of Month')
plt.ylabel('Predicted Revenue (Rs.)')
plt.xticks(range(1, 31, 2), [f'Day {i}' for i in range(1, 31, 2)], rotation=45)
plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/chart_30day_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Model verified and 30-day chart saved')